# WILDA Team 6 - Stage 2: Data Preparation
**Project:** Customer Churn Analysis for Telecommunication Company

This notebook covers missing-data checks, duplicate review, categorical encoding, an 80/20 stratified train/test split, and appropriate feature scaling. `Churn` is kept separate as the target.

In [1]:
import csv
from pathlib import Path
from collections import Counter
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

candidates = [Path.cwd() / "Dataset_ATS_v2.csv", Path.cwd().parent.parent / "Dataset_ATS_v2.csv"]
DATASET = next((p for p in candidates if p.exists()), None)
if DATASET is None:
    raise FileNotFoundError("Dataset_ATS_v2.csv was not found.")

with DATASET.open(newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    columns = reader.fieldnames

print("Rows:", len(rows))
print("Columns:", len(columns))
print("Column names:", columns)

Rows: 7043
Columns: 10
Column names: ['gender', 'SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'MonthlyCharges', 'Churn']


## 1. Data-quality checks
Missing values are checked before transformation. Exact duplicate-looking rows are counted but retained because the reduced dataset has no customer identifier; identical available attributes do not prove that two rows represent the same customer.

In [2]:
missing_by_column = {
    c: sum((r[c] is None) or (str(r[c]).strip() == "") for r in rows)
    for c in columns
}
duplicate_count = len(rows) - len({tuple(r[c] for c in columns) for r in rows})

print("Missing values:", missing_by_column)
print("Exact duplicate-looking rows:", duplicate_count)

Missing values: {'gender': 0, 'SeniorCitizen': 0, 'Dependents': 0, 'tenure': 0, 'PhoneService': 0, 'MultipleLines': 0, 'InternetService': 0, 'Contract': 0, 'MonthlyCharges': 0, 'Churn': 0}
Exact duplicate-looking rows: 302


## 2. Features and target
`Churn` is encoded as No = 0 and Yes = 1. Nominal categorical variables are one-hot encoded. `SeniorCitizen` is already a binary indicator, so it is retained as 0/1. The continuous variables `tenure` and `MonthlyCharges` are the variables that are standardised.

In [3]:
feature_cols = [c for c in columns if c != "Churn"]
categorical_cols = [
    "gender", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "Contract"
]
binary_numeric_cols = ["SeniorCitizen"]
continuous_cols = ["tenure", "MonthlyCharges"]

X = np.array([[
    r["gender"], int(r["SeniorCitizen"]), r["Dependents"], int(r["tenure"]),
    r["PhoneService"], r["MultipleLines"], r["InternetService"], r["Contract"],
    float(r["MonthlyCharges"])
] for r in rows], dtype=object)
y = np.array([1 if r["Churn"] == "Yes" else 0 for r in rows], dtype=int)

print("Target distribution:", Counter(y))

Target distribution: Counter({np.int64(0): 5174, np.int64(1): 1869})


## 3. 80/20 stratified split
The split uses `random_state=42` for reproducibility and `stratify=y` to preserve the churn distribution.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training records:", len(y_train), Counter(y_train))
print("Testing records:", len(y_test), Counter(y_test))

Training records: 5634 Counter({np.int64(0): 4139, np.int64(1): 1495})
Testing records: 1409 Counter({np.int64(0): 1035, np.int64(1): 374})


## 4. One-hot encoding and StandardScaler
The preprocessing object is fitted **only on the training data**. This avoids test-data leakage. `StandardScaler` is applied only to the continuous variables `tenure` and `MonthlyCharges`. `SeniorCitizen` remains a binary 0/1 indicator.

In [5]:
cat_idx = [feature_cols.index(c) for c in categorical_cols]
bin_idx = [feature_cols.index(c) for c in binary_numeric_cols]
cont_idx = [feature_cols.index(c) for c in continuous_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_idx),
        ("binary", "passthrough", bin_idx),
        ("continuous", StandardScaler(), cont_idx),
    ],
    remainder="drop"
)

X_train_processed = preprocessor.fit_transform(X_train).astype(float)
X_test_processed = preprocessor.transform(X_test).astype(float)

encoder = preprocessor.named_transformers_["cat"]
feature_names = []
for col, cats in zip(categorical_cols, encoder.categories_):
    feature_names.extend([f"{col}_{cat}" for cat in cats])
feature_names += binary_numeric_cols + continuous_cols

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)
print("Feature names:", feature_names)

Training shape: (5634, 16)
Testing shape: (1409, 16)
Feature names: ['gender_Female', 'gender_Male', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'SeniorCitizen', 'tenure', 'MonthlyCharges']


## 5. Validation
The checks below confirm that all rows are accounted for, the train and test feature structures match, and preprocessing introduced no invalid numeric values. The two scaled continuous training columns should have mean approximately 0 and standard deviation approximately 1.

In [6]:
assert len(y_train) + len(y_test) == len(y)
assert X_train_processed.shape[1] == X_test_processed.shape[1] == len(feature_names)
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()

scaled_train = X_train_processed[:, -2:]
print("Scaled continuous means:", scaled_train.mean(axis=0))
print("Scaled continuous standard deviations:", scaled_train.std(axis=0))
assert np.allclose(scaled_train.mean(axis=0), 0, atol=1e-10)
assert np.allclose(scaled_train.std(axis=0), 1, atol=1e-10)

print("All validation checks passed.")

Scaled continuous means: [1.44674829e-16 1.45980337e-16]
Scaled continuous standard deviations: [1. 1.]
All validation checks passed.


## Conclusion
The data is now prepared for Stage 2 analysis and modelling. No missing values were present, categorical variables were encoded, continuous variables were standardised without leakage, and the 80/20 stratified split is reproducible.